# reFuel to MOTEL unmapped records

This notebook structures the conversion of reFuel technology rows into MOTEL-compatible unmapped records.

## Goals
- Read reFuel input data from Excel.
- Load and inspect the MOTEL schema.
- Map each source row into a normalized record payload.
- Validate required fields and export records for review.

## Output target
- Write records to `dev2/motel-db/records/` as JSON or YAML files.

In [23]:
save_to_db = True

In [24]:
## --- load packages
from pathlib import Path
import pandas as pd
import yaml

# 1) Inputs and setup

## 1.1 Source data (reFuel Excel)

In [25]:
path_refuel = Path(r"E:\Barton\repositories\motel-platform\dev2\data\reFuel_TechDatabase_Clean_2026-06-03.xlsx")

refuel_data = pd.read_excel(path_refuel, sheet_name=None)
print(f"sheets found: {list(refuel_data.keys())}")

sheets found: ['Metadata', 'Nomenclature', 'Carrier', 'ConvTech', 'StorTech', 'EmbeddedCarbon', 'Reference']


## 1.2 Load MOTEL schema

In [26]:
path_schema = r'schema\motel_schema_rev2.yaml'

with open(path_schema, "r", encoding="utf-8") as f:
    schema_motel = yaml.safe_load(f)

print(f"Schema loaded successfully")
print(f"Type: {type(schema_motel)}")
if isinstance(schema_motel, dict):
    print(f"Top-level keys ({len(schema_motel)}): {list(schema_motel.keys())[:10]}")

Schema loaded successfully
Type: <class 'dict'>
Top-level keys (13): ['unmapped_record', 'mapped_record', 'technology', 'process', 'source', 'contributor', 'review', 'attribute', 'carrier', 'geographic_scope']


# 2) Create secondary data from Refuel.ch data

## 2.1 Technology datasets (checked)

source: sheet 'ConvTech'

NOTE: the process datasets may be decouple from the tech, since the process_id is out-of-date from the latest reFuel.ch data (Clean version).

In [27]:
sheet_name = 'ConvTech'
df_convtech = refuel_data[sheet_name].copy()

## correct column names
df_convtech.columns = df_convtech.loc[1]
df_convtech = df_convtech.loc[2:].reset_index(drop=True)

# deal with tech_category not in a column: add tech_category
df_convtech.insert(0, 'tech_category', pd.NA)
indx_filter = df_convtech[df_convtech['tech_type'].isna()].index

df_convtech.loc[indx_filter, 'tech_category'] = df_convtech.loc[indx_filter, 'tech_id']

df_convtech['tech_category'] = df_convtech['tech_category'].ffill(inplace=True)
df_convtech = df_convtech[df_convtech['tech_type'].notna()].reset_index(drop=True)

df_convtech

1,tech_category,tech_id,tech_year,technology_class,description,unit_operation,tech_type,reference_unit_size,cost_base,currency,...,opex_per_energy,discount_rate_pct,value_range_check,uncertainty_rating,mass_inflows,mass_outflows,energy_balance_error_pct,mass_balance_error_pct,balance_pass_flag,list_of_source_id
0,Electricity generation,NH3_CCGT_El,2050,Gas turbine,NaN,CCGT,Conversion,NaN,ECA,EUR,...,0.002,NaN,True,NaN,"1216, 1714",1000,0.6906,0.0122,True,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
1,Electricity generation,NH3_OCGT_El,2050,Gas turbine,NaN,OCGT,Conversion,NaN,ECA,EUR,...,0.002,NaN,True,NaN,"1216, 1714",1000,0.6906,0.0122,True,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
2,Electricity generation,CH4_CCGT_El,2050,Gas turbine,Methane-fueled CCGT with carbon capture and st...,CCGT,Conversion,NaN,ECA,EUR,...,0.002,NaN,True,Low,na,na,0.0600,0.0000,True,"""technical_efficiency"": paper_CCGT_2024; ""cap..."
3,Electricity generation,CH4_CHP_ElTh,2050,Combined heat & power,Combined heat and power (CHP) generation using...,CHP plant,Conversion,NaN,ECA,EUR,...,0.004,NaN,True,Low,na,na,0.0600,0.0000,True,"""capex_per_capacity"", ""opex_per_capacity_yr"",..."
4,Electricity generation,CH4_FuelCell_El,2030,Fuel cell,Methane-fueled fuel cell electricity generatio...,SOFC,Conversion,NaN,ECA,EUR,...,0,NaN,True,Medium,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,Carbon capture & sequestration,Air_DAC_CO2,2050,Carbon capture,Direct air capture using adsorption technology...,DAC (sorbent-based),Conversion,NaN,ECA,EUR,...,0,NaN,True,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
95,Carbon capture & sequestration,CO2Flue_PSA_CO2,2030,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,...,0.0022 EUR/kg,NaN,True,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
96,Carbon capture & sequestration,CO2Flue_PSA_CO2,2040,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,...,0.0022 EUR/kg,NaN,True,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."
97,Carbon capture & sequestration,CO2Flue_PSA_CO2,2050,Carbon capture,CO2 post combustion capture via Hot-Potassium ...,Pressure swing adsorption,Conversion,20,ECA,EUR,...,0.0022 EUR/kg,NaN,True,High,na,na,na,na,na,"""capex_per_capacity"", ""opex_per_capacity_yr"", ..."


In [28]:
def extract_conv_tech_structure_fast(df):
    # Clean column names
    df.columns = df.columns.str.strip()

    # Create the structured DataFrame directly via vectorised assignment
    result_df = pd.DataFrame()

    result_df["technology_name"] = df["tech_id"].astype(str).str.strip()
    result_df["technology_description"] = df["description"].astype(str).str.strip()
    result_df["technology_variant"] = df["unit_operation"].astype(str).str.strip()
    result_df["main_operation_unit"] = df["technology_class"].astype(str).str.strip()
    return result_df

df_technology = extract_conv_tech_structure_fast(df_convtech)

## remove duplicates
df_technology = df_technology.drop_duplicates(subset=['technology_name'], keep='first').reset_index(drop=True)

## add tech_id in format 'tech_00001'...
df_technology.insert(0, 'tech_id', ['tech_{:05d}'.format(i) for i in range(1, len(df_technology) + 1)])

df_technology

,tech_id,technology_name,technology_description,technology_variant,main_operation_unit
0,tech_00001,NH3_CCGT_El,NaN,CCGT,Gas turbine
1,tech_00002,NH3_OCGT_El,NaN,OCGT,Gas turbine
2,tech_00003,CH4_CCGT_El,Methane-fueled CCGT with carbon capture and st...,CCGT,Gas turbine
3,tech_00004,CH4_CHP_ElTh,Combined heat and power (CHP) generation using...,CHP plant,Combined heat & power
4,tech_00005,CH4_FuelCell_El,Methane-fueled fuel cell electricity generatio...,SOFC,Fuel cell
5,tech_00006,CH4_FuelCell_ElBdg,Methane-fueled fuel cell electricity generatio...,SOFC,Fuel cell
6,tech_00007,CH4_OCGT_El,Methane-fueled open cycle gas turbine (OCGT) e...,OCGT,Gas turbine
7,tech_00008,Geothermal_CHP_ElTh,Geothermal-based CHP generation (2040).,Geothermal power plant,Combined heat & power
8,tech_00009,H2_CCGT_El,Hydrogen-fueled combined cycle gas turbine (CC...,CCGT,Gas turbine
9,tech_00010,H2_FuelCell_El,Electricity generation via fuel cells at node ...,SOFC,Fuel cell


In [29]:
# export to csv

if save_to_db:
    df_technology.to_csv(r"motel-db\secondary\technology_export.csv", index=False)
    print(f'Technology data saved to CSV')

Technology data saved to CSV


## 2.2 Source (TODO)

In [30]:
# load source data from reFuel data (new structure uses sheet 'Reference')
df_ref = refuel_data["Reference"].copy()

# first row stores normalized header hints; keep only data rows
if len(df_ref) > 0 and str(df_ref.iloc[0].get("Source ID", "")).strip().lower() == "source_id":
    df_ref = df_ref.loc[1:].reset_index(drop=True)

# convert to motel-db source format
access_date_iso = pd.to_datetime(df_ref["Access Date"], errors="coerce").dt.strftime("%Y-%m-%d")
access_date_iso = access_date_iso.where(pd.notna(access_date_iso), None)

records_source = [
    {
        "source_name": row["Source ID"] if pd.notna(row["Source ID"]) else None,
        "source_description": row["Description"] if pd.notna(row["Description"]) else None,
        "source_type": row["Reference Type"] if pd.notna(row["Reference Type"]) else None,
        "link": (
            None
            if pd.isna(row["DOI or URL"]) or str(row["DOI or URL"]).strip().lower() in {"na", "nan", ""}
            else str(row["DOI or URL"]).strip()
        ),
        "access_date": access_date_iso.iloc[i],
        "confidence_level": None,
        "assessment_method": None,
        "assessment_date": None,
    }
    for i, row in df_ref.iterrows()
if pd.notna(row.get("Source ID")) and str(row.get("Source ID")).strip()
 ]

df_source = pd.DataFrame(records_source)
df_source.insert(0, "source_id", df_source.index.map(lambda x: f"src_{x+1:05d}"))

df_source

,source_id,source_name,source_description,source_type,link,access_date,confidence_level,assessment_method,assessment_date
0,src_00001,Allgoewer_2024,Paper by ETH from PATHFNDR,DOI,https://doi.org/10.1021/acs.iecr.4c01287,NaN,None,None,None
1,src_00002,Rheintal_2025,"Rheintal paper of Empa, submitted 2025",DOI,https://dx.doi.org/10.2139/ssrn.5291581,NaN,None,None,None
2,src_00003,biochar_jcp_2025,Financial feasibility of biochar production: A...,DOI,https://doi.org/10.1016/j.jclepro.2025.147167,NaN,None,None,None
3,src_00004,book_hydropower_2020,Future Energy (Third Edition),DOI,https://doi.org/10.1016/B978-0-08-102886-5.000...,NaN,None,None,None
4,src_00005,paper_CCGT+CCS_2016,Off-design point modelling of a 420 MW CCGT po...,DOI,https://doi.org/10.1016/j.apenergy.2016.06.087,NaN,None,None,None
5,src_00006,paper_CCGT_2024,Modeling combined-cycle power plants in a deta...,DOI,https://doi.org/10.1016/j.energy.2024.131246,NaN,None,None,None
6,src_00007,paper_Msw to syngas,Municipal Solid Waste Gasification: Technologi...,DOI,https://doi.org/10.3390/su17156704,NaN,None,None,None
7,src_00008,paper_PVcostProjection_2025,"review the cost projection of PV, wind, battery",DOI,https://doi.org/10.1016/j.apenergy.2025.125856,NaN,None,None,None
8,src_00009,paper_RWGS,Technology Readiness and Emerging Prospects of...,DOI,https://doi.org/10.1002/cssc.202400865,NaN,None,None,None
9,src_00010,paper_alpinePV_2024,ETHZ paper on Alpine PV cost and financial via...,DOI,https://doi.org/10.1016/j.apenergy.2024.124019,NaN,None,None,None


In [31]:
# export to csv

if save_to_db:
    df_source.to_csv(r"motel-db\secondary\source_export.csv", index=False)
    print(f'Source data saved to CSV')

Source data saved to CSV


## 2.3 Attribute (to be checked)

exclude attirbutes for balancing, deal with it for balancing later

In [32]:
## load the pre-defined mapping for attribute

path_attr_mapping = r"data\refuel2motel_map\attribute.csv"

df_attr_mapping = pd.read_csv(path_attr_mapping)
df_attr_mapping

,Raw Attribute,attribute_name,attribute_description,unit,data_format,attribute_note
0,technical_efficiency,technical_efficiency,Overall energy conversion efficiency: main out...,-,Float,Based on kW output capacity / LHV (chemical) e...
1,trl,technology_readiness_level,Technology Readiness Level on the standard 1–9...,-,Integer,See Nomenclature A-3.
2,tech_maturity,tech_maturity,Aggregated maturity classification derived fro...,-,String,See Nomenclature A-2.
3,reference_unit_size,reference_unit_size,Nameplate capacity on which all specific cost ...,CAP,Float,Capacity basis for scaling.
4,theoretical_efficiency,theoretical_efficiency,Ratio of the main output's energy content (LHV...,-,Float,May be blank if not applicable.
5,operating_temperature_c,operating_temperature,Typical operating temperature of the process.,°C,Float,Relevant for thermal integration.
6,min_installation_size,power_capacity_min,Smallest assumed unit capacity for the technol...,CAP,Float,0 if no minimum constraint.
7,lifetime_yr,lifetime_economic,Economic lifetime of the asset.,yr,Integer,Expected operational lifetime.
8,capex_one_time,capex_one_time,Fixed capital expenditure (independent of size).,CUR,Float,For project lump sum.
9,opex_fix_pct_of_capex,opex_fix_pct_of_capex,Fixed annual operational expenditure expressed...,%,Float,-


# 3) Create record from reFuel.ch data sheets

TODO: adept the conversion for balancing

## 3.1 Function Layers for Row -> Record Conversion

This notebook intentionally uses two function layers:

- Layer 1 (next code cell): base conversion logic from one reFuel row to one record, including final balancing schema construction.
- Layer 2 (following code cell): post-processing overrides that re-define selected functions (for example `add_value` and `map_technology`) to add filtering, unit parsing, and currency replacement.

Run the two code cells in order so Layer 2 extends Layer 1 correctly.

In [33]:
# LAYER 1 (BASE): core row -> record mapping.
# This cell defines the base implementation of map_technology and helpers.
# Balancing is built directly in final target schema here.
## --- function to convert reFuel.ch data into 'unmapped record'

import math
import re


def is_nan(value):
    return value is None or (
        isinstance(value, float) and math.isnan(value)
    )


def clean(value):
    return None if is_nan(value) else value


def parse_number_with_unit(text):
    """
    Converts:
        '2800 CHF/kg/h'
    into:
        (2800.0, 'CHF/kg/h')
    """
    if not isinstance(text, str):
        return text, None

    match = re.match(r"^\s*([0-9.]+)\s*(.*)$", text)
    if match:
        return float(match.group(1)), match.group(2).strip()

    return text, None


def split_csv(value):
    if is_nan(value):
        return []
    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip() and x.strip().lower() not in {"na", "nan"}
    ]


def split_csv_float(value):
    out = []
    for token in split_csv(value):
        try:
            out.append(float(token))
        except ValueError:
            out.append(None)
    return out


def split_number_unit(value):
    """Split strings like '20000 kg/h' into (20000.0, 'kg/h').

    Returns (original_value, None) when no numeric+unit pattern is found.
    """
    if not isinstance(value, str):
        return value, None

    text = value.strip()
    if not text:
        return value, None

    match = re.match(r"^([-+]?\d[\d,]*(?:\.\d+)?)\s+(.+)$", text)
    if not match:
        return value, None

    number_text, unit_text = match.groups()
    unit_text = unit_text.strip()
    if not unit_text:
        return value, None

    try:
        numeric_value = float(number_text.replace(",", ""))
    except ValueError:
        return value, None

    return numeric_value, unit_text


def is_empty_like_base(value):
    """Return True for empty / NA-like values used for final record pruning."""
    if is_nan(value):
        return True

    if isinstance(value, str):
        token = value.strip().lower()
        return token in {"", "na", "nan", "none", "null", "n/a"}

    if isinstance(value, (list, tuple, set)):
        return len(value) == 0 or all(is_empty_like_base(v) for v in value)

    if isinstance(value, dict):
        return len(value) == 0 or all(is_empty_like_base(v) for v in value.values())

    return False


def is_empty_like(value):
    """Return True for empty / NA-like values.

    Empty-like values include None, NaN, empty strings, and textual NA markers.
    Containers are considered empty when all contained values are empty-like.
    """
    if is_nan(value):
        return True

    if isinstance(value, str):
        token = value.strip().lower()
        return token in {"", "na", "nan", "none", "null", "n/a"}

    if isinstance(value, (list, tuple, set)):
        return len(value) == 0 or all(is_empty_like(v) for v in value)

    if isinstance(value, dict):
        return len(value) == 0 or all(is_empty_like(v) for v in value.values())

    return False


def add_value(values, name, description, value, unit=None, scenario=None, time_index=None):
    """Append one attribute if and only if value is non-empty.

    Also supports overriding default unit when value embeds unit text,
    e.g., value='20000 kg/h' with default unit='kW or kg/h'.
    """
    if is_empty_like(value):
        return

    parsed_value, parsed_unit = split_number_unit(value)
    if parsed_unit:
        value = parsed_value
        unit = parsed_unit

    record = {
        "attribute_name": name,
        "attribute_description": description,
        "unit": unit,
        "value": value,
    }

    if scenario:
        record["scenario"] = scenario

    if time_index:
        record["time_index"] = time_index

    values.append(record)


def build_balancing_entries(carriers, shares, units):
    size = max(len(carriers), len(shares), len(units))
    entries = []

    for i in range(size):
        carrier = carriers[i] if i < len(carriers) else None
        share = shares[i] if i < len(shares) else None
        unit = units[i] if i < len(units) else None

        if carrier is None and share is None and unit is None:
            continue

        entries.append({
            "carrier": carrier,
            "share": share,
            "unit": unit,
        })

    return entries


def to_balance_list(items):
    """Normalize balancing items to list of dicts with carrier_name/share/unit."""
    normalized = []
    for item in items or []:
        if not isinstance(item, dict):
            continue

        carrier_name = clean(item.get("carrier_name"))
        if carrier_name is None:
            carrier_name = clean(item.get("carrier"))

        share = clean(item.get("share"))
        unit = clean(item.get("unit"))

        if carrier_name is None and share is None and unit is None:
            continue

        normalized.append({
            "carrier_name": carrier_name,
            "share": share,
            "unit": unit,
        })

    return normalized


def prune_empty_keys(obj):
    """Recursively remove keys/items that are empty-like."""
    if isinstance(obj, dict):
        cleaned = {}
        for key, value in obj.items():
            pruned = prune_empty_keys(value)
            if not is_empty_like_base(pruned):
                cleaned[key] = pruned
        return cleaned

    if isinstance(obj, list):
        cleaned_list = []
        for value in obj:
            pruned = prune_empty_keys(value)
            if not is_empty_like_base(pruned):
                cleaned_list.append(pruned)
        return cleaned_list

    return obj


def normalize_unit(unit_text):
    if is_nan(unit_text):
        return None
    unit = str(unit_text).strip()
    if not unit or unit == "-":
        return None
    return unit


def split_or_units(unit_text):
    """Split expressions like 'kW or kg/h' into candidate units."""
    if not isinstance(unit_text, str):
        return []
    return [u.strip() for u in re.split(r"\s+or\s+", unit_text, flags=re.IGNORECASE) if u.strip()]


def infer_main_output_unit(main_output_carrier, output_carriers, output_units):
    """Infer the unit for the selected main output carrier."""
    if main_output_carrier is not None:
        for carrier, unit in zip(output_carriers, output_units):
            if clean(carrier) == main_output_carrier:
                unit_norm = normalize_unit(unit)
                if unit_norm:
                    return unit_norm

    # Fallback: first non-empty output unit if main output is not matched.
    for unit in output_units:
        unit_norm = normalize_unit(unit)
        if unit_norm:
            return unit_norm

    return None


def choose_preferred_or_unit(unit_text, main_output_unit):
    """Pick one unit from a disjunctive unit string using output-unit context."""
    candidates = split_or_units(unit_text)
    if not candidates:
        return normalize_unit(unit_text)

    if not main_output_unit:
        return candidates[0]

    target = str(main_output_unit).strip().lower()
    candidates_lower = [c.lower() for c in candidates]

    # Exact match first.
    for idx, cand_lower in enumerate(candidates_lower):
        if cand_lower == target:
            return candidates[idx]

    # Heuristics for common power vs mass-flow disambiguation.
    if ("kg/" in target) or target.startswith("kg") or ("ton/" in target) or target.startswith("t/"):
        for idx, cand_lower in enumerate(candidates_lower):
            if "kg/" in cand_lower or cand_lower.startswith("kg") or "ton/" in cand_lower or cand_lower.startswith("t/"):
                return candidates[idx]

    if ("w" in target) or ("wh" in target) or ("j/s" in target):
        for idx, cand_lower in enumerate(candidates_lower):
            if ("w" in cand_lower) or ("wh" in cand_lower) or ("j/s" in cand_lower):
                return candidates[idx]

    return candidates[0]


def resolve_attribute_unit(attribute_name, unit_text, main_output_unit):
    """Resolve attribute unit; supports context-based choice for 'A or B' units."""
    unit_norm = normalize_unit(unit_text)
    if not unit_norm:
        return None

    # Keep this generic so future attributes with disjunctive units also work.
    if isinstance(unit_norm, str) and re.search(r"\s+or\s+", unit_norm, flags=re.IGNORECASE):
        return choose_preferred_or_unit(unit_norm, main_output_unit)

    return unit_norm


def log_main_output_unit_resolution(tech_name, main_output_carrier, main_output_unit, specs):
    """Print a trace line showing inferred main-output unit and resolved capacity default."""
    capacity_unit_options = None
    for spec in specs:
        if spec.get("attribute_name") == "capacity_min":
            raw_unit = spec.get("unit")
            if isinstance(raw_unit, str) and re.search(r"\s+or\s+", raw_unit, flags=re.IGNORECASE):
                capacity_unit_options = raw_unit
                break

    if capacity_unit_options:
        selected_capacity_unit = choose_preferred_or_unit(capacity_unit_options, main_output_unit)
    else:
        selected_capacity_unit = None

    # print(
    #     f"[unit-resolution] tech '{tech_name}' has main_output '{main_output_carrier}' "
    #     f"with unit '{main_output_unit}', and default capacity unit '{selected_capacity_unit}' is used."
    # )


def build_attribute_specs(mapping_df):
    required_cols = {"Raw Attribute", "attribute_name", "attribute_description", "unit"}
    if not isinstance(mapping_df, pd.DataFrame) or not required_cols.issubset(mapping_df.columns):
        return []

    specs = []
    for _, rec in mapping_df.iterrows():
        raw_attr = clean(rec.get("Raw Attribute"))
        attr_name = clean(rec.get("attribute_name"))

        if not raw_attr or not attr_name:
            continue

        specs.append({
            "raw_attribute": str(raw_attr),
            "attribute_name": str(attr_name),
            "attribute_description": clean(rec.get("attribute_description")),
            "unit": normalize_unit(rec.get("unit")),
        })

    # De-duplicate by target attribute name, keep first mapping row.
    deduped = {}
    for spec in specs:
        deduped.setdefault(spec["attribute_name"], spec)

    return list(deduped.values())


def get_mapped_raw_value(row, raw_attribute, input_carriers, output_carriers):
    # This row in mapping is conceptual; derive from parsed carriers.
    if raw_attribute == "Input, Output for Overall Efficiency":
        combined = input_carriers + output_carriers
        return combined if combined else None

    return clean(row.get(raw_attribute))


def map_technology(row):

    currency = row['currency']    #!
    cap_unit = infer_main_output_unit(    #!
        clean(row.get("main_output")),      #!
        split_csv(row.get("carriers_out")),     #!
        split_csv(row.get("units_out_ratios")),     #!
    )

    result = {
        "technology_name": (
            row.get("tech_id", "")  #!
            .replace("_", " ")
        ),        

        "technology": {
    "technology_description": clean(row.get("description")),    #!
            "technology_type": clean(row.get("tech_type")),     #!
            "technology_category": clean(row.get("technology_class")),  #!
            "technology_assumption": None,
            "process_name": clean(row.get("unit_operation")),   #!
            "process_type": None,   #!
            "process_category": clean(row.get("tech_category")),    #!
            "process_assumption": None, #!
        },

        "scope": {
            "geographic_scope_description":
                clean(row.get("cost_base")),    #!
            "temporal_scope_description":   #!
                str(row.get("tech_year"))
                if not is_nan(row.get("tech_year"))
                else None,
            "capacity_scope_description":
                clean(row.get("min_installation_size")),
            "system_boundary_description":
                clean(row.get("tech_boundary")),
            "scope_assumption":
                clean(row.get("tech_maturity"))
        },

        "sources": [],
        "attributes": [],
        "balancing": {
            "inputs": [],
            "main_input": clean(row.get("main_input")),     #!
            "outputs": [],
            "main_output": clean(row.get("main_output")),      #!
            "balance_assumption": None,
        },
        "tags": {
            "related_project": "reFuel.ch",
            "tags": []
        }
    }

    attributes = result["attributes"]

    # -----------------------
    # Source parsing
    # -----------------------
    source_text = row.get("list_of_source_id")

    if isinstance(source_text, str):

        if "AECOM_2022" in source_text:
            result["sources"].append({
                "source_description": "AECOM_2022",
                "source_type": "report",
                "assessment_method": "literature"
            })

        if "parameters_lca" in source_text:
            result["sources"].append({
                "source_description": "parameters_lca",
                "source_type": "database",
                "assessment_method": "modeled"
            })

    # -----------------------
    # Inputs and outputs for balancing
    # -----------------------
    input_carriers = split_csv(row.get("carriers_in"))  #!
    input_shares = split_csv_float(row.get("ratios_in"))    #!
    input_units = split_csv(row.get("units_in_ratios"))     #!

    output_carriers = split_csv(row.get("carriers_out"))    #!
    output_shares = split_csv_float(row.get("ratios_out"))  #!
    output_units = split_csv(row.get("units_out_ratios"))   #!

    result["balancing"]["inputs"] = to_balance_list(
        build_balancing_entries(
            input_carriers,
            input_shares,
            input_units,
        )
    )
    result["balancing"]["outputs"] = to_balance_list(
        build_balancing_entries(
            output_carriers,
            output_shares,
            output_units,
        )
    )

    main_output_carrier = clean(row.get("main_output")) #!
    main_output_unit = infer_main_output_unit(
        main_output_carrier,
        output_carriers,
        output_units,
    )

    # -----------------------
    # Attributes from mapping CSV only
    # -----------------------
    specs = build_attribute_specs(df_attr_mapping)
    log_main_output_unit_resolution(
        result.get("technology_name"),
        main_output_carrier,
        main_output_unit,
        specs,
    )
    for spec in specs:
        value = get_mapped_raw_value(
            row,
            spec["raw_attribute"],
            input_carriers,
            output_carriers,
        )

        resolved_unit = resolve_attribute_unit(
            spec["attribute_name"],
            spec["unit"],
            main_output_unit,
        )

        add_value(
            attributes,
            spec["attribute_name"],
            spec["attribute_description"],
            value,
            unit=resolved_unit,
        )

    # -----------------------
    # Tags
    # -----------------------
    tags = result["tags"]["tags"]

    for field in [
        "main_sector",
        "main_category",
        "category_spec",
        "tech_type",
        "tech_maturity"
    ]:
        value = clean(row.get(field))
        if value:
            tags.append(str(value))

    return prune_empty_keys(result)

In [34]:
# LAYER 2 (OVERRIDE): compatibility helpers for updated reFuel structure.
# This cell wraps map_technology from Layer 1 and adapts renamed columns.

def _row_get(row, *keys):
    for key in keys:
        if key in row and not is_nan(row.get(key)):
            return row.get(key)
    return None


def _normalize_row_aliases(row):
    """Add legacy aliases expected by Layer 1 mapper when source columns were renamed."""
    row2 = row.copy()

    alias_map = {
        "Currency": "currency",
        "Cost Base": "cost_base",
        "cost_base_year": "tech_year",
        "main_out": "main_output",
        "units_in": "units_in_ratios",
        "units_out": "units_out_ratios",
        "main_category": "tech_category",
        "category_spec": "technology_class",
    }

    for legacy_key, new_key in alias_map.items():
        if legacy_key not in row2 or is_nan(row2.get(legacy_key)):
            if new_key in row2 and not is_nan(row2.get(new_key)):
                row2[legacy_key] = row2.get(new_key)

    return row2


def get_mapped_raw_value(row, raw_attribute, input_carriers, output_carriers):
    """Override with aliases for updated ConvTech raw attribute names."""
    if raw_attribute == "Input, Output for Overall Efficiency":
        combined = input_carriers + output_carriers
        return combined if combined else None

    alias_map = {
        "overall_efficiency": "technical_efficiency",
        "capex_one_time_eur": "capex_one_time",
        "capex_power_capacity_eur_per_kw": "capex_per_capacity",
        "opex_one_time_eur": "opex_one_time",
        "opex_fix_power_capacity_eur_per_kw_yr": "opex_per_capacity_yr",
        "opex_energy_eur_per_unit": "opex_per_energy",
    }

    value = clean(_row_get(row, raw_attribute, alias_map.get(raw_attribute, "")))
    return value


def apply_currency_to_units(record, row):
    """Replace CUR placeholder in attribute units with row-level currency."""
    currency = clean(_row_get(row, "Currency", "currency"))
    if not isinstance(currency, str):
        return record

    currency = currency.strip()
    if not currency:
        return record

    attr_list = record.get("attributes")
    if not isinstance(attr_list, list):
        attr_list = record.get("values", [])

    for item in attr_list:
        unit = item.get("unit")
        if isinstance(unit, str) and "CUR" in unit:
            item["unit"] = unit.replace("CUR", currency)

    return record


# Keep wrapper idempotent across re-runs by retaining a stable base function.
_current_map_technology = map_technology
if getattr(_current_map_technology, "_is_wrapper", False):
    _base_map_technology = _current_map_technology._base_map_technology
else:
    _base_map_technology = _current_map_technology


def map_technology(row):
    """Wrapper: normalize row aliases, build record, and apply unit post-processing."""
    row_norm = _normalize_row_aliases(row)
    rec = _base_map_technology(row_norm)
    rec = apply_currency_to_units(rec, row_norm)
    return rec


map_technology._is_wrapper = True
map_technology._base_map_technology = _base_map_technology

### Check the content of record

In [35]:
# run record creating for a testing row

def prepare_convtech_df(df_raw):
    """Normalize ConvTech table where row 1 contains machine-friendly headers."""
    df = df_raw.copy()
    header_idx = 1
    df.columns = [str(c).strip() for c in df.loc[header_idx]]
    df = df.loc[header_idx + 1:].reset_index(drop=True)

    # Remove section/header-like rows
    if "tech_id" in df.columns:
        df = df[df["tech_id"].notna()]
        df = df[df["tech_id"].astype(str).str.strip().str.lower() != "nan"]
        df = df[df["tech_id"].astype(str).str.strip().str.lower() != "tech_id"]
    if "unit_operation" in df.columns:
        df = df[df["unit_operation"].notna()]

    return df.reset_index(drop=True)

# load the data from ConvTech sheet for testing
df_conv = prepare_convtech_df(refuel_data["ConvTech"])

row = df_conv.loc[15]

# run record mapping for the testing row
record = map_technology(row)

In [36]:
# quick check: new balancing payload
record.get("balancing")

{'inputs': [{'carrier_name': 'HydroPumped', 'share': 1.0, 'unit': 'kWh'}],
 'main_input': 'HydroPumped',
 'outputs': [{'carrier_name': 'El13', 'share': 0.85, 'unit': 'kWh'}],
 'main_output': 'El13'}

In [37]:
# quick check: attributes come only from mapping CSV
mapped_attr_names = set(df_attr_mapping["attribute_name"].dropna().astype(str).str.strip())
record_attr_entries = record.get("attributes", record.get("values", []))
record_attr_names = {v["attribute_name"] for v in record_attr_entries if isinstance(v, dict) and "attribute_name" in v}

print("record attributes:", len(record_attr_names))
print("not in mapping:", sorted(record_attr_names - mapped_attr_names))

record attributes: 12
not in mapping: []


In [38]:
# quick check: no CUR placeholder should remain after currency replacement
record_attr_entries = record.get("attributes", record.get("values", []))
cur_units = [
    v.get("unit")
    for v in record_attr_entries
    if isinstance(v, dict) and isinstance(v.get("unit"), str) and "CUR" in v.get("unit")
]

print("remaining CUR units:", cur_units)
print("row currency:", row.get("currency", row.get("Currency")))

remaining CUR units: []
row currency: EUR


In [39]:
record.get("attributes", record.get("values", []))

[{'attribute_name': 'technical_efficiency',
  'attribute_description': 'Overall energy conversion efficiency: main output energy / main input energy.',
  'value': '0.85'},
 {'attribute_name': 'technology_readiness_level',
  'attribute_description': 'Technology Readiness Level on the standard 1–9 scale. For non-mature technologies, TRL values progress across tech_year.',
  'value': 9},
 {'attribute_name': 'tech_maturity',
  'attribute_description': 'Aggregated maturity classification derived from TRL. Updated per tech_year.',
  'value': 'Mature'},
 {'attribute_name': 'operating_temperature',
  'attribute_description': 'Typical operating temperature of the process.',
  'unit': '°C',
  'value': 0},
 {'attribute_name': 'power_capacity_min',
  'attribute_description': 'Smallest assumed unit capacity for the technology.',
  'unit': 'CAP',
  'value': 0},
 {'attribute_name': 'lifetime_economic',
  'attribute_description': 'Economic lifetime of the asset.',
  'unit': 'yr',
  'value': 80},
 {'at

## 3.2 Validate the format of record
check if the content in record match the schema, if there is missing setion or not...

in each the record

In [40]:
# Validate whether one record matches key schema expectations.
# This is written to support future user-provided records too.

def _is_missing(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def _collect_required_unit_attributes(attr_mapping_df):
    """Return attribute names whose unit is defined in mapping and should be present in record."""
    if not isinstance(attr_mapping_df, pd.DataFrame):
        return set()

    required = set()
    required_cols = {"attribute_name", "unit"}
    if not required_cols.issubset(attr_mapping_df.columns):
        return required

    for _, rec in attr_mapping_df.iterrows():
        name = rec.get("attribute_name")
        unit = rec.get("unit")
        if _is_missing(name) or _is_missing(unit):
            continue
        unit_text = str(unit).strip()
        if unit_text != "-":
            required.add(str(name).strip())
    return required


def validate_record_against_schema(record_obj, schema_obj=None, attr_mapping_df=None):
    """Validate record with practical rules derived from current schema and mapping.

    Rules checked:
    1) Required top-level sections exist.
    2) Required nested info exists (technology/scope/balancing minimum keys).
    3) attributes items have required keys (attribute_name, value).
    4) Unit is present when attribute unit is required by mapping.
    5) balancing items have carrier_name and share.
    """
    errors = []
    warnings = []

    if not isinstance(record_obj, dict):
        return {
            "valid": False,
            "errors": ["record must be a dictionary"],
            "warnings": [],
        }

    # Rule 1: required top-level keys
    required_top = [
        "technology_name",
        "technology",
        "scope",
        "attributes",
        "balancing",
    ]
    for key in required_top:
        if key not in record_obj:
            errors.append(f"missing top-level key: {key}")

    # Rule 2: required nested keys
    tech_obj = record_obj.get("technology", {})
    if isinstance(tech_obj, dict):
        for key in ["technology_type", "process_name"]:
            if _is_missing(tech_obj.get(key)):
                errors.append(f"missing technology.{key}")
    else:
        errors.append("technology must be a dictionary")

    scope_obj = record_obj.get("scope", {})
    if isinstance(scope_obj, dict):
        if _is_missing(scope_obj.get("geographic_scope_description")):
            warnings.append("scope.geographic_scope_description is empty")
    else:
        errors.append("scope must be a dictionary")

    bal_obj = record_obj.get("balancing", {})
    if isinstance(bal_obj, dict):
        if "inputs" not in bal_obj and "outputs" not in bal_obj:
            errors.append("balancing must have inputs and/or outputs")
        for side in ["inputs", "outputs"]:
            side_items = bal_obj.get(side, [])
            if side_items is None:
                side_items = []
            if not isinstance(side_items, list):
                errors.append(f"balancing.{side} must be a list")
                continue
            for i, item in enumerate(side_items):
                if not isinstance(item, dict):
                    errors.append(f"balancing.{side}[{i}] must be a dict")
                    continue
                if _is_missing(item.get("carrier_name")):
                    errors.append(f"balancing.{side}[{i}].carrier_name is missing")
                if _is_missing(item.get("share")):
                    errors.append(f"balancing.{side}[{i}].share is missing")
    else:
        errors.append("balancing must be a dictionary")

    # Rule 3 + 4: values structure + required unit when mapping says so
    required_unit_attrs = _collect_required_unit_attributes(attr_mapping_df)
    values_obj = record_obj.get("attributes", [])
    if not isinstance(values_obj, list):
        errors.append("attributes must be a list")
    else:
        seen_attr_names = set()
        for i, item in enumerate(values_obj):
            if not isinstance(item, dict):
                errors.append(f"attributes[{i}] must be a dict")
                continue

            attr_name = item.get("attribute_name")
            attr_value = item.get("value")
            attr_unit = item.get("unit")

            if _is_missing(attr_name):
                errors.append(f"attributes[{i}].attribute_name is missing")
                continue

            attr_name = str(attr_name).strip()
            seen_attr_names.add(attr_name)

            if attr_value is None:
                errors.append(f"attributes[{i}].value is missing for {attr_name}")

            if attr_name in required_unit_attrs and _is_missing(attr_unit):
                errors.append(f"attributes[{i}].unit is required for {attr_name}")

        if len(seen_attr_names) == 0:
            warnings.append("attribute list is empty")

    # Optional Rule 5: alignment with schema keys if schema is available
    if isinstance(schema_obj, dict):
        if "unmapped_record" not in schema_obj:
            warnings.append("schema does not contain unmapped_record section")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
    }

In [41]:
row

tech_id                                                Hydro_PumpedTurbine_El
tech_year                                                                2025
technology_class                                                   Hydropower
description                 Pumped hydroelectric power generation (turbine...
unit_operation                                    Pumped hydro (turbine mode)
tech_type                                                          Conversion
reference_unit_size                                                       NaN
cost_base                                                                 ECA
currency                                                                  EUR
trl                                                                         9
tech_maturity                                                          Mature
tech_boundary                                          Plant ready to operate
technical_efficiency                                            

In [42]:
## load the data from ConvTech sheet for testing
df_conv = refuel_data['ConvTech'].copy()
df_conv.columns = df_conv.loc[2]
df_conv = df_conv.loc[3:].reset_index(drop=True)

row = df_conv.loc[9]

## run record mapping for the testing row
record = map_technology(row)

## run record validation
validation_result = validate_record_against_schema(
    record_obj=record,
    schema_obj=schema_motel,
    attr_mapping_df=df_attr_mapping,
)

## show validation results
print("is_valid:", validation_result["valid"] )
print("error_count:", len(validation_result["errors"]))
print("warning_count:", len(validation_result["warnings"]))

if validation_result["errors"]:
    print("\nErrors:")
    for msg in validation_result["errors"][:20]:
        print("-", msg)

if validation_result["warnings"]:
    print("\nWarnings:")
    for msg in validation_result["warnings"][:20]:
        print("-", msg)

KeyError: 'currency'

## 3.3 Run record creating and validation for all reFuel.ch data

In [ ]:
## run mapping

records = []
sheet_name = "ConvTech"

# read row by row (per technology) using new ConvTech structure
if "prepare_convtech_df" in globals():
    df_conv = prepare_convtech_df(refuel_data[sheet_name])
else:
    df_conv = refuel_data[sheet_name].copy()
    df_conv.columns = [str(c).strip() for c in df_conv.loc[1]]
    df_conv = df_conv.loc[2:].reset_index(drop=True)

for indx in df_conv.index:
    row = df_conv.loc[indx]

    record = map_technology(row)
    print(f"Record for technology '{record.get('technology_name', '<missing>')}':")

    if "validate_record_against_schema" in globals():
        validation_result = validate_record_against_schema(
            record_obj=record,
            schema_obj=schema_motel,
            attr_mapping_df=df_attr_mapping,
        )
        if not validation_result["valid"]:
            print("  -> invalid:", len(validation_result["errors"]), "errors")

    records.append(record)

print("total records:", len(records))

In [ ]:
df_conv.loc[14]

In [ ]:
records
